<a href="https://colab.research.google.com/github/TaiwanOscarChen/Intelligent-index-interpretation/blob/main/%E6%99%BA%E8%83%BD%E5%88%A4%E8%AE%80%E6%8C%87%E6%95%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""智能判讀指數 - 訊號輸出自動化模擬版 (V106 核心升級，時區校正與盤中濾網版)"""

import os
import sys
import time
from datetime import datetime, timedelta
import antigravity
import google.generativeai as genai
# ... 準備對接您的策略與 GitHub

# ==========================================
# 【系統自動裝甲模組】開局自動檢查並安裝必備套件
# ==========================================
try:
    import pandas_ta as ta
except ImportError:
    print("⚙️ 偵測到環境缺乏 pandas_ta，系統正在為您自動安裝...")
    os.system(f"{sys.executable} -m pip install pandas_ta")
    import pandas_ta as ta

try:
    import pytz
except ImportError:
    os.system(f"{sys.executable} -m pip install pytz")
    import pytz

import pandas as pd
import numpy as np
import yfinance as yf
from IPython.display import display, HTML, clear_output

# [Google Drive 設定]
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    SAVE_PATH = "/content/drive/MyDrive/IronThrone_Log/"
    if not os.path.exists(SAVE_PATH):
        os.makedirs(SAVE_PATH)
except:
    SAVE_PATH = "./"

# [執行協議]: TX_HASH_V106_FULL_RESTORE_2026_AUTO_SIGNAL
# [規格]: 預算 20,000 | 0.3% 滑價 | V106動態移動停利 + 嚴格台股時區與盤中過濾

class IronThroneSmartTerminal:
    def __init__(self):
        self.initial_capital = 100000
        self.budget_per_trade = 20000
        self.base_profit_historical = 16.1
        self.start_date_analysis = "2025-10-01"
        self.target_year_start = "2026-01-01"
        self.ticker_map = {
            "00631L.TW": "元大台灣50正2",
            "2330.TW": "台積電",
            "1513.TW": "中興電",
            "1519.TW": "華城",
            "2382.TW": "廣達",
            "3017.TW": "奇鋐"
        }
        self.tickers = list(self.ticker_map.keys())
        self.tw_tz = pytz.timezone('Asia/Taipei') # ★ 設定台灣標準時間

    def fetch_and_analyze(self):
        data = yf.download(self.tickers, start=self.start_date_analysis, progress=False, auto_adjust=True)
        data_60m = yf.download(["00631L.TW"], start=self.start_date_analysis, interval="60m", progress=False, auto_adjust=True)

        # 🟢 YFinance 時區防呆校正 (將 UTC 強制轉回台灣時間)
        if data_60m.index.tz is None:
            data_60m.index = data_60m.index.tz_localize('UTC').tz_convert(self.tw_tz)
        else:
            data_60m.index = data_60m.index.tz_convert(self.tw_tz)

        # 拆股/除權息防呆修正模組
        if isinstance(data_60m.columns, pd.MultiIndex):
            df_60m = data_60m.xs('00631L.TW', axis=1, level=1).copy() if '00631L.TW' in data_60m.columns.levels[1] else data_60m.copy()
        else:
            df_60m = data_60m.copy()

        for i in range(1, len(df_60m)):
            if df_60m['Close'].iloc[i] < df_60m['Close'].iloc[i-1] * 0.6:
                split_ratio = round(df_60m['Close'].iloc[i-1] / df_60m['Close'].iloc[i])
                for col in ['Open', 'High', 'Low', 'Close']:
                    df_60m.loc[df_60m.index[:i], col] = df_60m[col].iloc[:i] / split_ratio

        # --- 1. 大盤紅綠燈分析 (總經 E-Stop) ---
        if isinstance(data.columns, pd.MultiIndex):
            tsmc = data['Close']['2330.TW']
        else:
            tsmc = data['Close']

        tsmc_ma20 = tsmc.rolling(20).mean()
        curr_tsmc = float(tsmc.iloc[-1])
        curr_tsmc_ma20 = float(tsmc_ma20.iloc[-1])
        is_bull = curr_tsmc > curr_tsmc_ma20
        market_mode = "Bull Dual (多頭雙進)" if is_bull else "Bear Single (空頭單進)"

        # --- 2. 主標的交易回溯 (00631L) - V106 核心 ---
        df_60m['20MA'] = ta.sma(df_60m['Close'], length=20)
        df_60m['10MA'] = ta.sma(df_60m['Close'], length=10)
        df_60m['ATR'] = ta.atr(df_60m['High'], df_60m['Low'], df_60m['Close'], length=14)
        df_60m['Hard_Stop'] = df_60m['20MA'] - (0.5 * df_60m['ATR'])
        df_60m['Vol_5MA'] = ta.sma(df_60m['Volume'], length=5)

        macd = ta.macd(df_60m['Close'], fast=12, slow=26, signal=9)
        if macd is not None and not macd.empty:
            df_60m['MACD_h'] = macd['MACDh_12_26_9']
            df_60m['MACD_val'] = macd['MACD_12_26_9']
        else:
            df_60m['MACD_h'] = 0
            df_60m['MACD_val'] = 0

        mask = df_60m.index >= pd.to_datetime(self.target_year_start).tz_localize(self.tw_tz)
        trade_df = df_60m[mask]

        trade_logs = []
        holdings = False
        tp_half_done = False
        buy_price = 0
        shares = 0

        for i in range(1, len(trade_df)):
            dt_obj = trade_df.index[i]
            time_val = dt_obj.hour * 100 + dt_obj.minute

            # ★ 盤中濾網：只抓取 09:00 ~ 13:30 之間的資料做交易觸發，排除盤後雜訊
            if not (900 <= time_val <= 1330):
                continue

            curr = trade_df.iloc[i]
            prev = trade_df.iloc[i-1]
            date_str = dt_obj.strftime('%Y-%m-%d %H:%M')

            price = float(curr['Close'])
            ma20 = float(curr['20MA'])
            ma10 = float(curr['10MA'])
            hard_stop = float(curr['Hard_Stop'])
            hist_curr = float(curr['MACD_h'])
            hist_prev = float(prev['MACD_h'])
            macd_val = float(curr['MACD_val'])
            vol = float(curr['Volume'])
            vol_5ma = float(curr['Vol_5MA'])

            if np.isnan(ma20) or np.isnan(hard_stop) or np.isnan(hist_curr): continue

            macd_turning_strong = (hist_curr > 0) and (hist_curr > hist_prev) and (macd_val > 0)
            vol_breakout = vol > (vol_5ma * 1.5)

            if not holdings and is_bull and price > ma20 and macd_turning_strong and vol_breakout:
                holdings = True
                tp_half_done = False
                buy_price = price
                shares = int(self.budget_per_trade // price)
                trade_logs.append({
                    "Time": f"{date_str}:00", "Code": "00631L",
                    "Action": "BUY", "Price": f"{price:.2f}", "Shares": f"{shares} 股",
                    "PnL": "-", "Reason": "V106: 站穩20MA+MACD擴張+爆量"
                })

            elif holdings:
                if price >= buy_price * 1.20 and not tp_half_done:
                    tp_half_done = True
                    sell_price = price * 0.997
                    sell_shares = shares // 2
                    shares -= sell_shares
                    pnl_val = ((sell_price - buy_price) / buy_price) * 100
                    trade_logs.append({
                        "Time": f"{date_str}:00", "Code": "00631L",
                        "Action": "SELL (TP)", "Price": f"{sell_price:.2f}", "Shares": f"{sell_shares} 股",
                        "PnL": f"{pnl_val:+.2f}%", "Reason": "🎯 獲利20%減碼一半，啟動10MA防守"
                    })

                defense_line = ma10 if tp_half_done else hard_stop
                if price < defense_line or price <= buy_price * 0.95:
                    sell_price = price * 0.997
                    pnl_val = ((sell_price - buy_price) / buy_price) * 100
                    reason_str = "🛡️ 跌破 10MA 移動停利結清" if tp_half_done and price < ma10 else "🛑 破 ATR防線/停損結清"
                    trade_logs.append({
                        "Time": f"{date_str}:00", "Code": "00631L",
                        "Action": "SELL (SL)", "Price": f"{sell_price:.2f}", "Shares": f"{shares} 股",
                        "PnL": f"{pnl_val:+.2f}%", "Reason": reason_str
                    })
                    holdings = False
                    tp_half_done = False

        # --- 3. 當前狀態與智能訊號 ---
        curr_last = trade_df.iloc[-1]
        prev_last = trade_df.iloc[-2]

        curr_price = float(curr_last['Close'])
        curr_ma20 = float(curr_last['20MA'])
        curr_ma10 = float(curr_last['10MA'])
        curr_hard_stop = float(curr_last['Hard_Stop'])
        active_defense = curr_ma10 if tp_half_done else curr_hard_stop

        prev_price = float(prev_last['Close'])
        momentum = (curr_price - prev_price) / prev_price * 100

        curr_hist = float(curr_last['MACD_h'])
        prev_hist = float(prev_last['MACD_h'])
        macd_val = float(curr_last['MACD_val'])
        vol = float(curr_last['Volume'])
        vol_5ma = float(curr_last['Vol_5MA'])

        macd_good = (curr_hist > 0) and (curr_hist > prev_hist) and (macd_val > 0)
        vol_breakout = vol > (vol_5ma * 1.5)

        if not holdings and is_bull and curr_price > curr_ma20 and macd_good and vol_breakout:
            signal_title = "🔴 強力買進 (STRONG BUY)"
            signal_desc = f"V106發動！價格站上20MA ({curr_ma20:.1f})，動能擴張且爆量。"
            bg_color = "#3d0e0e"
            txt_color = "#ff4d4d"
            action_required = "請準備【手動進場】：立即以市價建立基本部位。"
        elif not holdings and not is_bull and curr_price > curr_ma20 and macd_good and vol_breakout:
            signal_title = "⏸️ 大盤阻斷 (MACRO STOP)"
            signal_desc = f"個股訊號成形，但大盤(台積電)低於月線，觸發物理隔離。"
            bg_color = "#3d3d0e"
            txt_color = "#ffd700"
            action_required = "大盤環境不佳，建議暫緩狙擊。"
        elif holdings and curr_price < active_defense:
            signal_title = "🟢 立即出場 (URGENT EXIT)"
            signal_desc = f"趨勢轉折！價格跌破防守均線 ({active_defense:.1f})。"
            bg_color = "#0e3d16"
            txt_color = "#00e676"
            action_required = "請準備【手動平倉】：跌破防線，無條件市價變現。"
        elif holdings:
            safety_gap = ((curr_price - active_defense) / active_defense) * 100
            phase_str = "10MA 鎖利階段" if tp_half_done else "ATR 基礎防守"
            signal_title = "🟡 續抱持倉 (HOLD)"
            signal_desc = f"趨勢延續中 [{phase_str}]。距離破線還有 {safety_gap:.1f}% 空間。"
            bg_color = "#3d3d0e"
            txt_color = "#ffd700"
            action_required = "不需動作，持續監控跌破訊號。"
        else:
            signal_title = "⚪ 空手觀望 (WAIT)"
            signal_desc = f"趨勢未明。目前價格({curr_price:.1f})、動能或量能不足。"
            bg_color = "#1a1a1a"
            txt_color = "#888"
            action_required = "等待價格站上 20MA、MACD 翻紅且爆量1.5倍。"

        # --- 4. 權益與明日選股 ---
        perf_2026 = ((float(trade_df['Close'].iloc[-1]) - float(trade_df['Close'].iloc[0])) / float(trade_df['Close'].iloc[0])) * 100
        current_roi = self.base_profit_historical + perf_2026
        current_equity = self.initial_capital * (1 + (current_roi / 100))

        candidates = []
        for code, name in self.ticker_map.items():
            if code == "2330.TW": continue
            if code in data['Close']:
                s = data['Close'][code]
                s_ma5 = s.rolling(5).mean().iloc[-1]
                if float(s.iloc[-1]) > float(s_ma5):
                    score = (float(s.iloc[-1]) - float(s.iloc[-2])) / float(s.iloc[-2])
                    candidates.append({'label': f"{code} {name}", 'score': score})

        candidates.sort(key=lambda x: x['score'], reverse=True)
        top_picks = candidates[:(2 if is_bull else 1)]
        tomorrow_plan = [p['label'] for p in top_picks]

        # ★ 修正面板的更新時間為台灣時間
        now_tw = datetime.now(self.tw_tz)
        res = {
            "update_time": now_tw.strftime("%Y-%m-%d %H:%M:%S"),
            "tsmc_status": f"{curr_tsmc:.1f}",
            "tsmc_ma20": f"{curr_tsmc_ma20:.1f}",
            "is_bull": is_bull,
            "market_mode": market_mode,
            "main_price": f"{curr_price:.1f}",
            "main_ma5": f"{active_defense:.1f}",
            "momentum": f"{momentum:+.2f}%",
            "signal_title": signal_title,
            "signal_desc": signal_desc,
            "bg_color": bg_color,
            "txt_color": txt_color,
            "action_required": action_required,
            "tomorrow_plan": tomorrow_plan if tomorrow_plan else ["無符合規格標的"],
            "equity": current_equity,
            "roi": current_roi,
            "trade_logs": trade_logs
        }
        self.save_to_cloud(res)
        return res

    def save_to_cloud(self, res):
        now_tw = datetime.now(self.tw_tz)
        filename = f"V106_Full_Restore_{now_tw.strftime('%Y%m%d_%H%M')}.txt"
        file_path = os.path.join(SAVE_PATH, filename)
        log_str = "\n".join([f"{t['Time']} | {t['Action']} | {t['Price']} | {t['PnL']}" for t in res['trade_logs']])
        content = f"""[V106.0 智能決策快照]\n時間: {res['update_time']}\n訊號: {res['signal_title']}\n大盤: {res['market_mode']}\n權益: {int(res['equity']):,}\n明日計畫: {res['tomorrow_plan']}\n[交易紀錄]\n{log_str}"""
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(content)

    def render(self):
        res = self.fetch_and_analyze()
        tsmc_icon = "✅ 達標" if res['is_bull'] else "⚠️ 警戒"
        main_icon = "✅ 達標" if float(res['main_price']) > float(res['main_ma5']) else "❌ 未達標"

        log_rows = ""
        for t in reversed(res['trade_logs']):
            action_color = "#ff4d4d" if t['Action'] == "BUY" else "#00e676"
            pnl_color = "#ff4d4d" if "+" in t['PnL'] else ("#00e676" if "-" in t['PnL'] and t['PnL'] != "-" else "#eee")
            log_rows += f"""<tr style="border-bottom:1px solid #222;">
                <td style="padding:10px; color:#888;">{t['Time']}</td><td>{t['Code']}</td>
                <td style="color:{action_color}; font-weight:bold;">{t['Action']}</td>
                <td>{t['Price']}</td><td style="color:#d4af37;">{t['Shares']}</td>
                <td style="color:{pnl_color};">{t['PnL']}</td><td style="color:#666; font-size:0.85em;">{t['Reason']}</td>
            </tr>"""

        # 原版介面，完全無變動
        html = f"""
        <div style="background:#050505; color:#e0e0e0; padding:25px; border:4px solid {res['txt_color']}; border-radius:15px; font-family:'Microsoft JhengHei', sans-serif; max-width:900px; margin:auto; box-shadow: 0 0 20px {res['txt_color']}40;">
            <div style="text-align:center; border-bottom:1px solid #333; padding-bottom:15px; margin-bottom:20px;">
                <h1 style="margin:0; font-size:2.2em; letter-spacing:3px; color:#d4af37;">智能判讀指數</h1>
                <p style="color:#888; font-size:0.9em; margin-top:5px;">更新時間：{res['update_time']} | 策略模式：手動實單訊號監控</p>
            </div>
            <div style="background:{res['bg_color']}; padding:25px; border-radius:12px; text-align:center; margin-bottom:25px; border:2px solid {res['txt_color']};">
                <div style="font-size:1.2em; color:#ddd; margin-bottom:10px;">❗ V106 智能判讀結果 (Current Signal)</div>
                <div style="font-size:3em; font-weight:900; color:{res['txt_color']}; text-shadow: 0 0 10px {res['txt_color']}; margin:10px 0;">{res['signal_title']}</div>
                <div style="font-size:1.3em; font-weight:bold; color:#fff; margin-bottom:15px;">{res['signal_desc']}</div>
                <div style="background:#000; display:inline-block; padding:10px 20px; border-radius:50px; border:1px solid {res['txt_color']}; color:{res['txt_color']}; font-weight:bold;">
                    ⚡ 執行指令：{res['action_required']}
                </div>
            </div>
            <div style="display:grid; grid-template-columns: 1fr 1fr; gap:20px; margin-bottom:25px;">
                <div style="background:#111; padding:15px; border-radius:10px; border:1px solid #333;">
                    <div style="color:#aaa; font-weight:bold; border-bottom:1px solid #444; padding-bottom:8px; margin-bottom:10px;">1. 大盤紅綠燈 (2330)</div>
                    <div style="display:flex; justify-content:space-between;"><span>規格 (Price > 20MA)</span><span style="font-weight:bold;">{tsmc_icon}</span></div>
                    <div style="font-size:0.9em; color:#888;">台積電現價: <span style="color:#fff;">{res['tsmc_status']}</span></div>
                    <div style="margin-top:10px; font-weight:bold; color:{'#00e676' if res['is_bull'] else '#ff4d4d'}; text-align:right;">{res['market_mode']}</div>
                </div>
                <div style="background:#111; padding:15px; border-radius:10px; border:1px solid #333;">
                    <div style="color:#aaa; font-weight:bold; border-bottom:1px solid #444; padding-bottom:8px; margin-bottom:10px;">2. 00631L 規格 (60m級別)</div>
                    <div style="display:flex; justify-content:space-between;"><span>動態防線 (Price > 趨勢線)</span><span style="font-weight:bold;">{main_icon}</span></div>
                    <div style="font-size:0.9em; color:#888;">當前保護線: <span style="color:#d4af37;">{res['main_ma5']}</span></div>
                    <div style="margin-top:10px; text-align:right;"><span style="color:#aaa; font-size:0.8em;">動能強度:</span> <span style="color:{'#ff4d4d' if '+' in res['momentum'] else '#00e676'}; font-weight:bold;">{res['momentum']}</span></div>
                </div>
            </div>
            <div style="display:grid; grid-template-columns: 1.5fr 1fr; gap:20px; margin-bottom:25px;">
                <div style="background:linear-gradient(45deg, #1a1a1a, #000); padding:20px; border:1px solid #d4af37; border-radius:10px; text-align:center;">
                    <div style="color:#d4af37; font-weight:bold; margin-bottom:10px;">🎯 明日作戰計畫</div>
                    <div style="font-size:1.8em; font-weight:900; color:#fff;">{" / ".join(res['tomorrow_plan'])}</div>
                </div>
                <div style="display:grid; grid-template-rows: 1fr 1fr; gap:10px;">
                    <div style="background:#000; border:1px solid #333; text-align:center; padding:10px;"><small style="color:#666;">當前權益</small><br><b style="font-size:1.4em; color:#fff;">${int(res['equity']):,}</b></div>
                    <div style="background:#000; border:1px solid #333; text-align:center; padding:10px;"><small style="color:#666;">累積報酬</small><br><b style="font-size:1.4em; color:#ff4d4d;">+{res['roi']:.1f}%</b></div>
                </div>
            </div>
            <div style="background:#111; border:1px solid #333; border-radius:10px; padding:15px; margin-bottom:20px;">
                <div style="color:#d4af37; font-weight:bold; margin-bottom:10px; border-bottom:1px solid #333;">📋 交易紀錄 (進出場明細) - 滑動檢視</div>
                <div style="max-height: 250px; overflow-y: auto;">
                    <table style="width:100%; border-collapse:collapse; color:#eee; text-align:left; font-size:0.9em;">
                        <thead style="position: sticky; top: 0; background: #111;">
                            <tr style="color:#666; background:#000;">
                                <th style="padding:10px;">執行時間</th><th>標的</th><th>動作</th><th>執行價</th><th>股數</th><th>損益</th><th>理由</th>
                            </tr>
                        </thead>
                        <tbody>{log_rows}</tbody>
                    </table>
                </div>
            </div>
            <div style="background:#1a1a1a; padding:15px; border-left:5px solid #d4af37; font-size:0.9em; line-height:1.6;">
                <b style="color:#d4af37;">🦁 V106 策略規格說明 (盤中實戰版)：</b><br>
                1. <b>大盤紅綠燈</b>：以 2330 台積電為大盤強弱指標。站上 20MA 開放雙倍投資，跌破則發動物理隔離停買。<br>
                2. <b>60m 動能發動</b>：當價格向上穿透 20MA 且 MACD 動能擴張、突破 1.5倍爆量時，觸發買進訊號。<br>
                3. <b>動態 ATR 防護</b>：跌破「20MA - 0.5ATR」防禦線或跌 5% 強制停損。<br>
                4. <b>時間濾網校正</b>：已導入 pytz 台灣時間校正，交易與訊號判定<b>嚴格鎖定於 09:00 ~ 13:30 盤中執行</b>。<br>
            </div>
        </div>
        """
        display(HTML(html))

    def start_auto_simulation(self, interval_seconds=300):
        print("🚀 獅王戰神 V106 - 啟動全自動訊號監控模式 (準備進入無限迴圈)")
        while True:
            try:
                clear_output(wait=True) # 清除上一次的畫面，實現在地刷新
                self.render()
                print(f"\n⏳ 下次戰情掃描將在 {interval_seconds} 秒後自動執行... (請保持 Colab 網頁開啟)")
                print("🛑 若要停止監控，請手動點擊左側停止按鈕 (Stop)。")
                time.sleep(interval_seconds)
            except KeyboardInterrupt:
                print("\n🛑 戰情室監控已由操作員手動停止。")
                break
            except Exception as e:
                print(f"\n⚠️ 抓取報價時發生異常: {e}")
                print("系統將在 60 秒後重試...")
                time.sleep(60)

# 執行
v106_smart = IronThroneSmartTerminal()
v106_smart.start_auto_simulation(interval_seconds=300) # 預設每 5 分鐘 (300秒) 更新一次

/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:147: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end_dt = pd.Timestamp.utcnow().tz_convert(tz)
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:147: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end_dt = pd.Timestamp.utcnow().tz_convert(tz)
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/usr/local/lib/python3.12/dist-packages/yfinance/scrap

執行時間,標的,動作,執行價,股數,損益,理由
2026-05-22 10:00:00,00631L,BUY,31.94,626 股,-,V106: 站穩20MA+MACD擴張+爆量
2026-05-12 09:00:00,00631L,SELL (SL),32.16,647 股,+4.16%,🛑 破 ATR防線/停損結清
2026-05-04 10:00:00,00631L,BUY,30.88,647 股,-,V106: 站穩20MA+MACD擴張+爆量
2026-04-28 12:00:00,00631L,SELL (SL),29.44,436 股,+28.45%,🛡️ 跌破 10MA 移動停利結清
2026-04-24 09:00:00,00631L,SELL (TP),28.09,436 股,+22.54%,🎯 獲利20%減碼一半，啟動10MA防守
2026-04-08 10:00:00,00631L,BUY,22.92,872 股,-,V106: 站穩20MA+MACD擴張+爆量
2026-03-19 12:00:00,00631L,SELL (SL),20.52,966 股,-0.90%,🛑 破 ATR防線/停損結清
2026-03-17 10:00:00,00631L,BUY,20.70,966 股,-,V106: 站穩20MA+MACD擴張+爆量
2026-03-03 09:00:00,00631L,SELL (SL),22.20,982 股,+9.10%,🛑 破 ATR防線/停損結清
2026-02-10 11:00:00,00631L,BUY,20.35,982 股,-,V106: 站穩20MA+MACD擴張+爆量



⏳ 下次戰情掃描將在 300 秒後自動執行... (請保持 Colab 網頁開啟)
🛑 若要停止監控，請手動點擊左側停止按鈕 (Stop)。
